# 03 – Ensemble Methods: Gradient Boosting & Stacking

Ensembles combine multiple models to reduce error.  
This notebook covers the three main strategies:

| Strategy | Idea | Example |
|----------|------|---------|
| **Bagging** | Parallel trees, reduce variance | Random Forest |
| **Boosting** | Sequential trees, reduce bias | GBM, XGBoost, LightGBM |
| **Stacking** | Train a meta-model on predictions | `StackingClassifier` |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    StackingClassifier,
    VotingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

%matplotlib inline
sns.set_theme(style='whitegrid')
np.random.seed(42)

In [ ]:
bc = load_breast_cancer()
X, y = bc.data, bc.target

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc  = scaler.transform(X_te)

## 1. AdaBoost

In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # stumps
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)
ada.fit(X_tr_sc, y_tr)

print('AdaBoost Test Accuracy:', accuracy_score(y_te, ada.predict(X_te_sc)):.3f)

# Staged accuracy (how accuracy evolves with more estimators)
staged_acc = [accuracy_score(y_te, pred) for pred in ada.staged_predict(X_te_sc)]
plt.figure(figsize=(8, 4))
plt.plot(staged_acc, color='steelblue')
plt.xlabel('Number of estimators'); plt.ylabel('Accuracy')
plt.title('AdaBoost – Staged Accuracy'); plt.show()

## 2. Gradient Boosting Machine (sklearn)

In [ ]:
gbm = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
)
gbm.fit(X_tr_sc, y_tr)

print('GBM Test Accuracy:', accuracy_score(y_te, gbm.predict(X_te_sc)):.3f)
print('GBM Test AUC     :', roc_auc_score(y_te, gbm.predict_proba(X_te_sc)[:, 1]):.3f)

In [ ]:
# Feature importance
feat_imp = pd.Series(gbm.feature_importances_, index=bc.feature_names).nlargest(15)
feat_imp.sort_values().plot(kind='barh', figsize=(8, 5), color='coral')
plt.title('GBM – Top 15 Feature Importances'); plt.tight_layout(); plt.show()

## 3. XGBoost

In [ ]:
try:
    import xgboost as xgb
    print('XGBoost version:', xgb.__version__)

    xgb_clf = xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    xgb_clf.fit(X_tr_sc, y_tr,
                eval_set=[(X_te_sc, y_te)],
                verbose=False)

    print('XGBoost Test Accuracy:', accuracy_score(y_te, xgb_clf.predict(X_te_sc)):.3f)
    print('XGBoost Test AUC     :', roc_auc_score(y_te, xgb_clf.predict_proba(X_te_sc)[:, 1]):.3f)

except ImportError:
    print('XGBoost not installed. Run: pip install xgboost')

## 4. LightGBM

In [ ]:
try:
    import lightgbm as lgb
    print('LightGBM version:', lgb.__version__)

    lgb_clf = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=31,
        subsample=0.8,
        random_state=42,
        verbose=-1
    )
    lgb_clf.fit(X_tr_sc, y_tr,
                eval_set=[(X_te_sc, y_te)],
                callbacks=[lgb.early_stopping(30, verbose=False)])

    print('LightGBM Test Accuracy:', accuracy_score(y_te, lgb_clf.predict(X_te_sc)):.3f)
    print('LightGBM Test AUC     :', roc_auc_score(y_te, lgb_clf.predict_proba(X_te_sc)[:, 1]):.3f)

except ImportError:
    print('LightGBM not installed. Run: pip install lightgbm')

## 5. Stacking & Voting

In [ ]:
# Voting classifier (hard & soft)
estimators = [
    ('rf',  RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gbm', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('lr',  LogisticRegression(max_iter=1000))
]

hard_vote = VotingClassifier(estimators=estimators, voting='hard')
soft_vote = VotingClassifier(estimators=estimators, voting='soft')

for name, vc in [('Hard Voting', hard_vote), ('Soft Voting', soft_vote)]:
    vc.fit(X_tr_sc, y_tr)
    print(f'{name} Test Accuracy: {accuracy_score(y_te, vc.predict(X_te_sc)):.3f}')

In [ ]:
# Stacking
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    passthrough=False,
    cv=5
)
stack.fit(X_tr_sc, y_tr)
print('Stacking Test Accuracy:', accuracy_score(y_te, stack.predict(X_te_sc)):.3f)

In [ ]:
# Summary comparison
results = {
    'Random Forest':     accuracy_score(y_te, RandomForestClassifier(100, random_state=42).fit(X_tr_sc, y_tr).predict(X_te_sc)),
    'AdaBoost':          accuracy_score(y_te, ada.predict(X_te_sc)),
    'GBM':               accuracy_score(y_te, gbm.predict(X_te_sc)),
    'Soft Voting':       accuracy_score(y_te, soft_vote.predict(X_te_sc)),
    'Stacking':          accuracy_score(y_te, stack.predict(X_te_sc))
}

pd.Series(results).sort_values().plot(kind='barh', figsize=(8, 4), color='steelblue')
plt.xlim(0.9, 1.0)
plt.title('Model Comparison – Test Accuracy'); plt.tight_layout(); plt.show()

## 6. Key Takeaways

| Model | Strength |
|-------|----------|
| AdaBoost | Simple, interpretable boosting |
| GBM | Strong accuracy, tunable |
| XGBoost | Fast, regularised GBM |
| LightGBM | Fastest on large datasets |
| Stacking | Learns how to combine base models |

**Next:** `04_NLP_Basics.ipynb`